In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/ayushikhandelia/tables/tables.json
/kaggle/input/datasets/ayushikhandelia/spider-dataset-train/train_spider.json


In [4]:
import json

# # with open("/kaggle/input/datasets/ayushikhandelia/spider-dataset/train_others.json") as f:
#     data = json.load(f)

# print(len(data))
# print(data[0])

In [5]:
!pip install -q transformers peft bitsandbytes datasets accelerate sqlparse

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.2 MB/s eta 0:00:00:00:0100:01


In [6]:
import torch
import json
import sqlparse
import sqlite3
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model

In [7]:
model_name = "mistralai/Mistral-7B-v0.1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [8]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

In [9]:
# import json

# with open("/kaggle/input/spider-dataset/train_spider.json") as f:
#     raw = json.load(f)

In [10]:
data = []

with open("/kaggle/input/datasets/ayushikhandelia/spider-dataset-train/train_spider.json", "r") as f:
    raw = json.load(f)

In [11]:
with open("/kaggle/input/datasets/ayushikhandelia/tables/tables.json") as f:
    tables = json.load(f)

In [12]:
db_schemas = {}

for table in tables:
    db_id = table["db_id"]
    
    table_names = table["table_names_original"]
    column_names = table["column_names_original"]
    
    schema = {}

    for table_id, col_name in column_names:
        if table_id == -1:
            continue
        
        table_name = table_names[table_id]
        
        if table_name not in schema:
            schema[table_name] = []
        
        schema[table_name].append(col_name)

    db_schemas[db_id] = schema

In [13]:
def build_schema(db):
    schema = "Tables:\n"
    
    for table, columns in db.items():
        schema += f"{table}({', '.join(columns)})\n"
    
    return schema

In [14]:
clean_data = []

for ex in raw[:3000]:
    schema_dict = db_schemas[ex["db_id"]]
    schema = build_schema(schema_dict)

    prompt = f"""You are an expert SQL generator.

Database Schema:
{schema}

Question:
{ex['question']}

SQL Query:
{ex['query']}"""

    clean_data.append({"text": prompt})

In [15]:
dataset = Dataset.from_list(clean_data)

In [16]:
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=256,
        padding="max_length"
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

dataset = dataset.map(tokenize)

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [17]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/sql-genie",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="no",
    save_strategy="no",
    fp16=True,
    report_to="none"
)

In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()

Step,Training Loss
10,2.245158
20,0.796064
30,0.562051
40,0.443710
50,0.391531
60,0.324745
70,0.301379
80,0.257848
90,0.210931
100,0.215074


TrainOutput(global_step=375, training_loss=0.2623379850387573, metrics={'train_runtime': 3347.099, 'train_samples_per_second': 0.896, 'train_steps_per_second': 0.112, 'total_flos': 3.2781625196544e+16, 'train_loss': 0.2623379850387573, 'epoch': 1.0})

In [19]:
def is_valid_sql(query):
    try:
        sqlparse.parse(query)
        return True
    except:
        return False


def schema_check(query, schema):
    cols = schema.lower().split()
    for word in query.lower().split():
        if word.isalpha() and word not in cols:
            return False
    return True

In [20]:
def optimize_query(query):
    query = query.strip()

    # 🔹 Remove repeated spaces
    query = " ".join(query.split())

    # 🔹 Remove SELECT *
    if "select *" in query.lower():
        query = query.replace("*", "id")

    # 🔹 Normalize operators
    query = query.replace(" = ", "=")
    query = query.replace(" > ", ">")
    query = query.replace(" < ", "<")

    # 🔹 Remove duplicate WHERE conditions
    if "where" in query.lower():
        parts = query.split("WHERE")
        conditions = parts[1].split("AND")
        conditions = list(set([c.strip() for c in conditions]))
        query = parts[0] + "WHERE " + " AND ".join(conditions)

    return query

In [21]:
def generate_sql(schema, question):
    prompt = f"""You are an expert SQL generator.

Database Schema:
{schema}

Generate a SQL query.

Question:
{question}

SQL Query:"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    output = model.generate(
        **inputs,
        max_new_tokens=120,
        temperature=0.1,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    sql = tokenizer.decode(output[0], skip_special_tokens=True)

    # 🔹 APPLY OPTIMIZATION
    optimized_sql = optimize_query(sql)

    return optimized_sql

In [22]:
schema = """
Tables:
students(id, name, age)
marks(student_id, subject, score)

Foreign Keys:
marks.student_id = students.id
"""

question = "List names of students who scored more than 90 in Math"

sql = generate_sql(schema, question)

print(sql)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


You are an expert SQL generator. Database Schema: Tables: students(id, name, age) marks(student_id, subject, score) Foreign Keys: marks.student_id=students.id Generate a SQL query. Question: List names of students who scored more than 90 in Math SQL Query: SELECT T1.name FROM students AS T1 JOIN marks AS T2 ON T1.id=T2.student_id WHERE T2.score>90 AND T2.subject="Math"


In [23]:
print("Valid SQL:", is_valid_sql(sql))
print("Schema Valid:", schema_check(sql, schema))

Valid SQL: True
Schema Valid: False


In [24]:
# SAVE LoRA adapter
model.save_pretrained("/kaggle/working/sql-genie-lora")
tokenizer.save_pretrained("/kaggle/working/sql-genie-lora")

print("Saved LoRA adapter!")

Saved LoRA adapter!


In [26]:
from peft import PeftModel

model = PeftModel.from_pretrained(
    base_model,
    "/kaggle/working/sql-genie-lora",
    local_files_only=True   
)